# YOLO 3회차 산업 데모 ④ — 통행량과 속도: 주차장 출입 관제

위에서 내려다보는 카메라로 차량·자전거·보행자를 추적해 **기준선을 몇 대가 어느 방향으로 지나갔는지(통행량)** 와 **얼마나 빨리 움직였는지(속도)** 를 구합니다.
속도는 화면 위 이동 거리(픽셀)를 실제 거리(미터)로 바꿔야 나옵니다. 이 노트북은 **크기를 아는 물체(승용차 길이)를 자로 삼는 방법**으로 환산하고, 보행자의 걸음 속도로 검산합니다.

| 단계 | 하는 일 | 결과물 |
|---|---|---|
| Step 1 | 차량·자전거·사람 탐지 + 추적 | 관측 표 `obs` |
| Step 2 | 픽셀 → 미터 환산 (승용차 길이를 자로) | `PX_PER_M` |
| Step 3 | 속도 계산과 검산 | `traffic_tracks.csv` |
| Step 4 | 규칙 — 기준선 통행량, 서행 위반 | `traffic_counts.csv` |
| Step 5 | 관제 영상과 리포트 | `traffic_monitor.mp4`, `traffic_report.png` |

**실습 영상**: Intel IoT DevKit 공개 샘플 `person-bicycle-car-detection.mp4` (주차장을 위에서 찍은 영상, 768×432, 12fps, 54초, CC BY 4.0 — 출처: github.com/intel-iot-devkit/sample-videos)
**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**(Colab에서 빌려 쓰는 NVIDIA 그래픽카드)

## 0. 준비

In [ ]:
# 패키지 설치 — ultralytics(YOLO), lap(ByteTrack이 박스를 짝지을 때 사용). -q: 설치 로그 줄임
!pip -q install ultralytics lap

In [ ]:
import os, json, shutil, base64, subprocess, time
from pathlib import Path

# 영상·이미지 처리(cv2), 배열(numpy), 표(pandas), 그래프(matplotlib)
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display  # 셀 출력에 HTML(영상 플레이어 등) 띄우기
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())  # T4 런타임이면 True

try:
    from google.colab import drive  # Colab에만 있는 모듈 → 성공하면 Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True  # False면 Drive 대신 런타임 임시 폴더에 저장 (연결이 끊기면 사라짐)
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day3/traffic')  # Drive 안 결과 폴더
else:
    SAVE_DIR = Path('outputs/traffic')
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 중간 폴더까지 생성, 이미 있으면 그냥 통과
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# OpenCV(BGR) 이미지를 matplotlib로 보여 주기
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 색 축을 뒤집어 BGR→RGB, 눈금 숨김
    if title:
        plt.title(title)
    plt.show()

# 여러 장을 격자로 보여 주기 (입력: BGR 이미지 목록, 제목 목록, 한 줄에 놓을 장 수)
def show_grid(imgs, titles=None, cols=3, size=4.5):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 줄 수 (올림)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for k, im in enumerate(imgs):
        ax = axes.ravel()[k]
        ax.imshow(im[..., ::-1])
        if titles:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout(); plt.show()

# mp4 영상을 노트북 안에서 재생 (입력: 영상 경로, 표시 폭 px)
# OpenCV가 만든 mp4v 영상은 브라우저가 못 틀어서 H.264로 다시 인코딩
def show_video(path, width=640):
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()  # 영상을 글자(base64)로 바꿔 HTML에 직접 넣음
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

# 파일을 SAVE_DIR로 복사하고 복사된 경로를 돌려줌
def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 원본과 대상이 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

In [ ]:
# 실습 영상 내려받기 (Intel IoT DevKit 공개 샘플, CC BY 4.0)
VIDEO = 'parking_lot.mp4'  # 영문 파일 이름으로 저장
if not Path(VIDEO).exists():
    !wget -q https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4 -O {VIDEO}

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)  # 초당 프레임 수 (이 영상은 12)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, FIRST = cap.read()
cap.release()
print(f'{VIDEO}: {W}x{H}, {FPS:.0f}fps, {N_FRAMES}프레임, {N_FRAMES / FPS:.1f}초')
show_bgr(FIRST, 'first frame')

---
## Step 1. 차량·자전거·사람 탐지 + 추적

COCO 사전학습 모델(`yolo26n.pt`, 80종 물체로 미리 학습된 모델)에서 통행 관제에 필요한 클래스만 남깁니다: 사람(0), 자전거(1), 승용차(2), 오토바이(3), 버스(5), 트럭(7).
추적(tracking — 프레임이 바뀌어도 같은 물체에 같은 번호 track_id를 붙이는 기술)을 켜야 "같은 차가 얼마나 이동했는지"를 계산할 수 있습니다.

In [ ]:
model = YOLO('yolo26n.pt')
CLASSES = [0, 1, 2, 3, 5, 7]  # person, bicycle, car, motorcycle, bus, truck
CONF = 0.3                    # 신뢰도(conf — 모델이 매긴 0~1 확신 점수) 기준

rows = []
# stream=True: 프레임마다 결과를 하나씩 받음, persist=True: 추적 정보를 다음 프레임으로 이어 감
for k, r in enumerate(model.track(VIDEO, stream=True, persist=True, classes=CLASSES, conf=CONF,
                                  tracker='bytetrack.yaml', verbose=False)):
    if r.boxes.id is None:
        continue
    for (x1, y1, x2, y2), tid, c, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                            r.boxes.cls.int().cpu().tolist(), r.boxes.conf.cpu().numpy()):
        rows.append({'frame': k, 'time_s': round(k / FPS, 3), 'track_id': tid, 'cls_name': model.names[c],
                     'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                     'cx': (x1 + x2) / 2, 'cy': (y1 + y2) / 2, 'conf': cf})  # cx, cy: 박스 중심
obs = pd.DataFrame(rows)
# 한 track_id에 클래스가 여러 개로 나오면(자전거 ↔ 사람 등) 가장 많이 나온 클래스로 통일
major = obs.groupby('track_id')['cls_name'].agg(lambda s: s.value_counts().index[0])
obs['cls_name'] = obs['track_id'].map(major)
print('관측', len(obs), '줄')
obs.groupby('cls_name')['track_id'].nunique().rename('tracks')

위에서 내려다본 차는 COCO 학습 사진(대부분 옆·앞에서 찍은 차)과 모양이 달라 신뢰도가 낮게 나오거나 다른 클래스로 헷갈릴 수 있습니다. 실제 현장에서는 이런 각도의 사진으로 미세조정(사전학습 모델을 내 데이터로 조금 더 학습시키는 것)을 합니다.

---
## Step 2. 픽셀 → 미터 환산 — 승용차 길이를 자로 쓰기

속도 = 이동 거리 ÷ 시간입니다. 영상에서 재는 거리는 **픽셀**이므로 "1미터가 몇 픽셀인가(`PX_PER_M`)"를 알아야 합니다.
현장에서 쓰는 방법은 크게 세 가지입니다.
1. 바닥에 길이를 아는 표시(주차선, 차선 간격)를 찍어 잰다
2. **크기를 아는 물체**(일반 승용차 길이 약 4.7m)를 자로 쓴다 ← 이 노트북
3. 카메라 설치 높이·각도로 계산한다(원근 보정, homography)

차가 화면에서 가장 길게 보이는 길이(박스의 긴 변)의 중앙값을 승용차 길이로 나눕니다. 이 카메라는 거의 수직으로 내려다보므로 화면 위치에 따른 크기 차이(원근)는 작다고 보고 하나의 값으로 씁니다.

In [ ]:
CAR_LEN_M = 4.7  # ✏️ 일반 승용차 길이(미터) — 이 값이 자의 눈금
cars = obs[obs['cls_name'] == 'car'].copy()
cars['long_px'] = np.maximum(cars.x2 - cars.x1, cars.y2 - cars.y1)  # 박스의 긴 변(픽셀)
# 화면 가장자리에 걸려 잘린 박스는 실제보다 짧으므로 제외
inside = (cars.x1 > 3) & (cars.y1 > 3) & (cars.x2 < W - 3) & (cars.y2 < H - 3)
ruler = cars[inside].groupby('track_id')['long_px'].median()  # 차마다 긴 변 중앙값
PX_PER_M = float(ruler.median() / CAR_LEN_M) if len(ruler) else 20.0  # 차가 안 잡히면 임시값 20
print('차량별 긴 변(px):', ruler.round(1).to_dict())
print(f'PX_PER_M = {PX_PER_M:.1f} px/m  (1픽셀 ≈ {100 / PX_PER_M:.1f} cm)')

---
## Step 3. 속도 계산과 검산

프레임마다 박스 중심이 조금씩 흔들리므로 바로 옆 프레임끼리 빼면 속도가 크게 튑니다. `SMOOTH_S`초(예: 0.5초) **떨어진 두 위치**로 이동 거리를 재면 흔들림이 줄어듭니다.
물체마다 순간 속도들의 **중앙값**(대표 속도)과 **90번째 백분위수**(가끔 튀는 값을 빼고 본 최고 속도)를 구합니다.

In [ ]:
SMOOTH_S = 0.5                         # 속도를 재는 시간 간격(초)
LAG = max(1, int(round(SMOOTH_S * FPS)))  # 그 간격이 몇 프레임인가

obs = obs.sort_values(['track_id', 'frame']).reset_index(drop=True)
g = obs.groupby('track_id')
# shift(LAG): 같은 track_id 안에서 LAG 줄 앞의 값 → 0.5초 전 위치
dx = obs['cx'] - g['cx'].shift(LAG); dy = obs['cy'] - g['cy'].shift(LAG)
dt = obs['time_s'] - g['time_s'].shift(LAG)
obs['speed_kmh'] = (np.hypot(dx, dy) / PX_PER_M / dt * 3.6).round(1)  # m/s × 3.6 = km/h

tracks = obs.groupby('track_id').agg(cls_name=('cls_name', 'first'), first_s=('time_s', 'min'), last_s=('time_s', 'max'),
                                     n_obs=('frame', 'size'), median_kmh=('speed_kmh', 'median'),
                                     p90_kmh=('speed_kmh', lambda s: s.quantile(0.9))).round(1).reset_index()
tracks = tracks[tracks['n_obs'] >= LAG * 2]  # 너무 짧게 잡힌 번호는 속도를 믿기 어려워 제외
tracks

In [ ]:
# 검산: 걷는 사람의 대표 속도는 보통 4~6 km/h. 크게 다르면 PX_PER_M(자 눈금)을 의심
walk = tracks[tracks['cls_name'] == 'person']['median_kmh']
print('보행자 대표 속도(km/h):', walk.tolist())
if len(walk):
    print(f'→ 중앙값 {walk.median():.1f} km/h', '(그럴듯함)' if 2.5 <= walk.median() <= 8 else '(환산값 다시 확인 필요)')

**생각해 보기** — 보행자 속도가 그럴듯하게 나왔다면, 승용차 길이로 만든 자가 대략 맞다는 증거입니다. 크기를 아는 물체 두 가지(차, 사람 걸음)로 서로 검산하는 습관이 현장 측정에서 중요합니다.

---
## Step 4. 규칙 — 기준선 통행량과 서행 위반

**통행량**: 화면에 가로 기준선(`LINE_Y`)을 긋고, 물체 중심이 선을 위→아래(`down`) 또는 아래→위(`up`)로 넘은 횟수를 클래스별로 셉니다. 1회차 박스 카운팅과 같은 원리입니다.
**서행 위반**: 주차장 안 차량은 보통 시속 10km 이하 서행입니다. 차량·오토바이의 90번째 백분위 속도가 `SPEED_LIMIT_KMH`를 넘으면 위반으로 표시합니다.

In [ ]:
LINE_Y = int(H * 0.55)      # ✏️ 기준선 높이(픽셀) — 화면 세로의 55% 위치
SPEED_LIMIT_KMH = 10        # ✏️ 서행 기준(km/h)
VEHICLES = ['car', 'motorcycle', 'bus', 'truck']

# 기준선 넘기: 같은 track_id에서 '앞 프레임 cy'와 '지금 cy'가 선의 양쪽에 있으면 넘은 것
obs['prev_cy'] = obs.groupby('track_id')['cy'].shift(1)
down = (obs['prev_cy'] < LINE_Y) & (obs['cy'] >= LINE_Y)
up = (obs['prev_cy'] >= LINE_Y) & (obs['cy'] < LINE_Y)
cross = pd.concat([obs[down].assign(direction='down'), obs[up].assign(direction='up')])
cross = cross.drop_duplicates(['track_id', 'direction'])  # 선 근처에서 흔들려 여러 번 넘은 것은 한 번으로

counts = cross.groupby(['cls_name', 'direction']).size().unstack(fill_value=0)
counts.to_csv('traffic_counts.csv'); save_out('traffic_counts.csv')  # 열 이름은 영문
tracks['crossed'] = tracks['track_id'].isin(cross['track_id'])
tracks['speeding'] = tracks['cls_name'].isin(VEHICLES) & (tracks['p90_kmh'] > SPEED_LIMIT_KMH)
tracks.to_csv('traffic_tracks.csv', index=False); save_out('traffic_tracks.csv')
print('기준선 통행량'); display(counts)
print('서행 위반 차량'); tracks[tracks['speeding']]

---
## Step 5. 관제 영상과 리포트

In [ ]:
# 관제 영상: 물체마다 번호·클래스·현재 속도, 기준선과 누적 통행량 표시
by_frame = {f: d for f, d in obs.groupby('frame')}
speeding_ids = set(tracks.loc[tracks['speeding'], 'track_id'])
cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter('traffic_monitor.mp4', cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
k = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    passed = cross[cross['frame'] <= k]  # 지금까지 넘은 기록
    cv2.line(frame, (0, LINE_Y), (W, LINE_Y), (0, 255, 255), 2)
    cv2.putText(frame, f'down {int((passed.direction == "down").sum())}  up {int((passed.direction == "up").sum())}',
                (10, LINE_Y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    for _, o in by_frame.get(k, obs.iloc[:0]).iterrows():
        col = (0, 0, 255) if o.track_id in speeding_ids else (0, 200, 0)  # 위반 빨강, 정상 초록
        cv2.rectangle(frame, (int(o.x1), int(o.y1)), (int(o.x2), int(o.y2)), col, 2)
        sp = '' if np.isnan(o.speed_kmh) else f' {o.speed_kmh:.0f}km/h'
        cv2.putText(frame, f'#{o.track_id} {o.cls_name}{sp}', (int(o.x1), max(15, int(o.y1) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    cv2.putText(frame, f't={k / FPS:.1f}s  1m={PX_PER_M:.0f}px', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    out.write(frame)
    k += 1
cap.release(); out.release()
save_out('traffic_monitor.mp4')
show_video('traffic_monitor.mp4')

In [ ]:
# 리포트: 왼쪽 = 클래스별 통행량, 오른쪽 = 물체별 속도(중앙값~90백분위)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot.bar(ax=axes[0], rot=0); axes[0].set_title(f'line crossings (y={LINE_Y}px)'); axes[0].grid(alpha=.3, axis='y')
axes[0].yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # 눈금을 정수(대수)로만
t = tracks.sort_values('p90_kmh')
labels = [f'#{i} {c}' for i, c in zip(t.track_id, t.cls_name)]
axes[1].barh(labels, t['p90_kmh'], color=['tab:red' if s else 'tab:gray' for s in t['speeding']], alpha=.6, label='p90')
axes[1].scatter(t['median_kmh'], labels, c='k', s=15, label='median', zorder=3)
axes[1].axvline(SPEED_LIMIT_KMH, ls='--', c='red', label=f'limit {SPEED_LIMIT_KMH} km/h')
axes[1].set_xlabel('km/h'); axes[1].legend(); axes[1].set_title(f'speed by track (1 m = {PX_PER_M:.1f} px)')
plt.tight_layout(); plt.savefig('traffic_report.png', dpi=120); save_out('traffic_report.png'); plt.show()
print(sorted(p.name for p in SAVE_DIR.iterdir()))

---
## 도전 과제

1. **주차선으로 자 만들기** — 첫 화면에서 주차선 두 개 사이 간격(보통 2.5m)을 픽셀로 재서 `PX_PER_M`을 다시 구하고, 승용차로 만든 값과 비교해 보세요.
2. **도로 영상으로** — 차로가 있는 도로 영상으로 바꾸고 기준선을 차로마다 하나씩 그어 차로별 통행량을 세어 보세요. Ultralytics의 `solutions.SpeedEstimator`, `solutions.ObjectCounter`와 결과를 비교해 보는 것도 좋습니다.
3. **체류 차량** — 주차면 구역을 다각형으로 그리고, 차가 구역 안에 멈춰 있는 시간으로 "주차 중 / 이동 중"을 나눠 보세요(2회차 위험구역 실습의 체류 시간 규칙).

## 정리

- 통행량은 **추적 + 기준선 넘기**, 속도는 **추적 + 픽셀→미터 환산**으로 구합니다.
- 환산값(`PX_PER_M`)은 크기를 아는 물체로 만들고, **다른 물체로 검산**합니다.
- 프레임 간 흔들림 때문에 속도는 일정 시간 간격과 중앙값·백분위수로 안정적으로 구합니다.
- 비스듬히 찍은 카메라는 화면 위치마다 1m의 픽셀 수가 달라 원근 보정이 필요합니다. 이 영상은 거의 수직이라 하나의 값으로 근사했습니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| 추적과 track_id | 프레임이 바뀌어도 같은 물체에 같은 번호를 붙이는 기술과 그 번호입니다. |
| ByteTrack | Ultralytics가 제공하는 추적 방식입니다. |
| COCO 클래스 | COCO 데이터셋의 80개 물체 종류입니다. 사람 0, 자전거 1, 승용차 2, 오토바이 3, 버스 5, 트럭 7입니다. |
| 박스 중심 (cx, cy) | 박스 가운데 점입니다. 위에서 내려다본 영상에서는 물체 위치로 씁니다. |
| PX_PER_M | 1미터가 화면에서 몇 픽셀인지 나타내는 환산값입니다. 속도 계산의 자 눈금입니다. |
| 원근 / homography | 비스듬히 찍으면 먼 곳이 작게 보이는 현상 / 화면 좌표를 바닥 평면 좌표로 바꾸는 변환입니다. |
| shift | pandas에서 표의 값을 몇 줄 밀어 '이전 값'을 옆에 놓는 함수입니다. 같은 track_id 안에서 이동 거리를 구할 때 씁니다. |
| km/h 환산 | 초속(m/s)에 3.6을 곱하면 시속(km/h)입니다. |
| 중앙값 / 90번째 백분위수 | 값을 크기순으로 줄 세웠을 때 가운데 값 / 아래에서 90% 위치 값입니다. 튀는 값의 영향을 줄입니다. |
| 기준선 넘기 | 앞 프레임과 지금 프레임의 위치가 선의 양쪽에 있으면 선을 넘은 것으로 봅니다. |
| 서행 위반 | 차량의 90번째 백분위 속도가 기준 속도를 넘은 경우입니다. |
| 미세조정 | 사전학습 모델을 내 데이터로 조금 더 학습시키는 것입니다. |
| conf (신뢰도) | 모델이 예측마다 매긴 0~1 확신 점수입니다. |
| fps | 초당 프레임 수입니다. 이 영상은 12fps입니다. |
| H.264 / ffmpeg | 브라우저가 재생할 수 있는 영상 압축 방식 / 영상 형식을 바꾸는 명령줄 도구입니다. |